# TutorTask72 Wildlife Image Classification
**Tool:** Custom JAX/Flax workflow | **Dataset:** Animals-10

In this notebook we take in a small subset of the data set, configure the CNN, run a tiny training loop, and read the evaluation metrics that come back. 
- **What you will see:** Every code cell produces human-readable output so you can sanity-check shapes, hyper-parameters, and metrics.
- **What you can tweak:** Point `DATA_DIR` to any Animals-10 location, change the image size, or cap the number of samples per class to keep turnaround time manageable.
- **Deliverable tie-in:** The same APIs power the larger notebook, the markdown docs, and the Docker workflow.


## Section 1 - Imports and Configuration
The first cell wires up `logging`, pulls in `load_dataset`, `TrainConfig`, `train`, and `evaluate`, and defines the knobs we adjust most often:
- `DATA_DIR` points at `./data/animals10`, which matches the repo layout described in the README.
- `IMAGE_SIZE` is fixed at 128x128 RGB because that resolution balances detail with memory usage.
- `LIMIT_PER_CLASS = 50` keeps this lightweight; on my machine it loads in a couple of seconds.


# JAX_wildlife.API

## Section 2 - Load and Inspect the Dataset
Running `load_dataset` immediately prints two facts: the project still sees **10 classes** and the tensors come back with shapes `{'train': (350, 128, 128, 3), 'val': (75, ...), 'test': (75, ...)}`. That tells us:
1. Images were resized to 128x128 RGB and normalized to floats in `[0,1]`.
2. The 50-per-class cap produced 350 training examples plus balanced validation/test splits, so every species is represented even in this tiny subset.
3. The helper respected the `(0.7, 0.15, 0.15)` split we passed in. If any of these numbers looked off, this is the cell where we would catch it.


In [1]:
import logging
from JAX_wildlife_utils import load_dataset, TrainConfig, train, evaluate

logging.basicConfig(level=logging.INFO)
DATA_DIR = './data/animals10'  
IMAGE_SIZE = (128, 128)
LIMIT_PER_CLASS = 50  


## Section 3 - Configure and Train SimpleCNN
Here we instantiate `TrainConfig` with one epoch, a batch size of 64, and a learning rate of `1e-3`. Those values match the defaults in `JAX_wildlife_utils.py`: three convolutional blocks feeding a dense layer with dropout. The INFO logs that stream by (`Epoch 1/1 loss=... acc=...`) confirm that:
- Parameters initialize correctly on CPU-only hardware.
- Optax Adam steps execute without numerical issues.
- `history` captures loss and accuracy so we can graph them later if needed.
Because we only train for a single epoch on a handful of images, the goal is correctness, not accuracy.


In [2]:
Xs, ys, class_names = load_dataset(
    DATA_DIR, image_size=IMAGE_SIZE, splits=(0.7, 0.15, 0.15), limit_per_class=LIMIT_PER_CLASS
)
len(class_names), {split: arr.shape for split, arr in Xs.items()}

(10,
 {'train': (350, 128, 128, 3),
  'val': (75, 128, 128, 3),
  'test': (75, 128, 128, 3)})

## Section 4 - Evaluate on the Test Split
The last cell calls `evaluate` and prints a metrics dictionary. On the 50-per-class subset, the model settles around **10% accuracy** with matching precision/recall because it has seen so little data. The confusion matrix shows every prediction landing in one column, which is exactly what we expect from a barely trained classifier. 


In [3]:
config = TrainConfig(image_size=IMAGE_SIZE, num_classes=len(class_names), num_epochs=1, batch_size=64)
state, history = train(Xs['train'], ys['train'], Xs['val'], ys['val'], config)
metrics = evaluate(state, Xs['test'], ys['test'], class_names)
metrics

INFO:2025-12-09 20:10:32,034:jax._src.xla_bridge:752: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
INFO:JAX_wildlife_utils:Epoch 1/1 loss=3.2495 acc=0.1359 val_acc=0.0933


{'accuracy': 0.10666666666666667,
 'precision': 0.010666666666666668,
 'recall': 0.1,
 'confusion_matrix': array([[ 0,  0,  0, 10,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  7,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0, 10,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  8,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  4,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  6,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  9,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  8,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  6,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  7,  0,  0,  0,  0,  0,  0]]),
 'y_pred': array([3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
        3, 3, 3, 3, 3, 3, 3, 3, 3], dtype=int32)}